# Inserting csv's to Neo4J

In [57]:
# !pip install neo4j
!git status

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python3 -m pip install --upgrade pip
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   11_token_distribution.ipynb
	modified:   13_llm_experiment.ipynb
	modified:   5_graph.ipynb
	modified:   7_all_retriever_routes.ipynb
	modified:   8_evaluate_2.ipynb
	deleted:    create_dataset.py
	deleted:    evalute.py
	deleted:    pickles/filtered_testset.pkl
	deleted:    pickles/merged_testset_9300.pkl
	deleted:    pickles/processed_dataset.pkl
	deleted:    pickles/progress.log
	deleted:    pickles/ya_allah.pkl
	modified:   utils/prompt/main.py

Untracked files:
  (use "git add <file>..." to include in what will be committed

In [28]:
# !git clone https://github.com/ninggar17/Lexid.git
# !cd Lexid/

In [6]:
NEO4J_URI="neo4j+s://afd9a6f6.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="6NMK9rluWNgznfhzbQmCdgfhsON1UXczctpscFoStu4"
AURA_INSTANCEID="afd9a6f6"
AURA_INSTANCENAME="Instance01"

In [12]:
TYPE = 'mencabut'
LAW = 'uu'
RELATIONSHIP_TYPE = TYPE.upper()

In [8]:
from py2neo import Graph
from utils.config import get_config

config = get_config()

graph = Graph(NEO4J_URI, name=NEO4J_USERNAME, password=NEO4J_PASSWORD)

In [13]:
import pandas as pd

df_combined = pd.read_csv(f"csv/{LAW}/{TYPE}.csv")

In [14]:
print(len(df_combined))
df_combined.head(1)

532


,PERATURAN,STATUS,RELATED_PERATURAN,PERATURAN_LINK,RELATED_PERATURAN_LINK,PERATURAN_jenis_peraturan,PERATURAN_pemrakarsa,PERATURAN_nomor,PERATURAN_tahun,PERATURAN_tentang,PERATURAN_tempat_penetapan,PERATURAN_status,RELATED_jenis_peraturan,RELATED_pemrakarsa,RELATED_nomor,RELATED_tahun,RELATED_tentang,RELATED_tempat_penetapan,RELATED_status
0,uu-no-13-tahun-2003,Mencabut,uu-no-25-tahun-1997,https://peraturan.go.id/id/uu-no-13-tahun-2003,https://peraturan.go.id/id/uu-no-25-tahun-1997,UNDANG-UNDANG,PEMERINTAH PUSAT,13,2003,KETENAGAKERJAAN,Jakarta,Berlaku,UNDANG-UNDANG,PEMERINTAH PUSAT,25,1997,KETENAGAKERJAAN,Jakarta,Tidak Berlaku Dicabut Oleh :Undang-Undang Nomo...


In [15]:
from py2neo import Node, Relationship

nodes = {}

# Function to create nodes
def create_node(label, properties):
    node_id = properties['id']
    if node_id not in nodes:
        node = Node(label, **properties)
        graph.merge(node, label, "id")
        nodes[node_id] = node
    return nodes[node_id]

# Function to create relationships
def create_relationship(source_id, target_id, relationship_type):
    source_node = nodes.get(source_id)
    target_node = nodes.get(target_id)
    if source_node and target_node:
        relationship = Relationship(source_node, relationship_type, target_node)
        graph.merge(relationship)

# Create the reverse relationship
def create_reverse_relationship(source_id, target_id, relationship_type):
    if relationship_type == "MENCABUT":
        reverse_relationship_type = "DICABUT"
    elif relationship_type == "MENGUBAH":
        reverse_relationship_type = "DIUBAH"
    else:
        raise ValueError(f"Unknown relationship type: {relationship_type}")
    
    create_relationship(target_id, source_id, reverse_relationship_type)


# Iterate over the DataFrame and create nodes and relationships
for index, row in df_combined.iterrows():
    # Create a node for PERATURAN
    peraturan_properties = {
        "id": row['PERATURAN_LINK'],
        "nomor": row['PERATURAN_nomor'],
        "jenis_peraturan": row['PERATURAN_jenis_peraturan'],
        "pemrakarsa": row['PERATURAN_pemrakarsa'],
        "tahun": row['PERATURAN_tahun'],
        "tentang": row['PERATURAN_tentang'],
        "tempat_penetapan": row['PERATURAN_tempat_penetapan'],
        "status": row['PERATURAN_status'],
    }
    peraturan_node = create_node("Peraturan", peraturan_properties)

    # Create a node for RELATED_PERATURAN
    related_properties = {
        "id": row['RELATED_PERATURAN_LINK'],
        "nomor": row['RELATED_nomor'],
        "jenis_peraturan": row['RELATED_jenis_peraturan'],
        "pemrakarsa": row['RELATED_pemrakarsa'],
        "tahun": row['RELATED_tahun'],
        "tentang": row['RELATED_tentang'],
        "tempat_penetapan": row['RELATED_tempat_penetapan'],
        "status": row['RELATED_status'],
    }
    related_node = create_node("Peraturan", related_properties)

    create_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)
    create_reverse_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)

## Import Lexid-KG to GraphDB

In [64]:
ENDPOINT_URL = 'https://8d21-2a09-bac5-55ff-1028-00-19c-16f.ngrok-free.app/repositories/Lexid-KG-UU'
UPDATE_URL = 'https://8d21-2a09-bac5-55ff-1028-00-19c-16f.ngrok-free.app/repositories/Lexid-KG-UU/statements'

In [65]:
headers = {
    'Content-Type': 'text/turtle',
    'Accept': 'application/json'
}

In [87]:
TYPE = 'mencabut'
LAW = 'uu'
RELATIONSHIP_TYPE = TYPE.upper()

In [88]:
import pandas as pd

df_combined = pd.read_csv(f"csv/{LAW}/{TYPE}.csv")

In [90]:
print(len(df_combined))
df_combined.head(1)

532


,PERATURAN,STATUS,RELATED_PERATURAN,PERATURAN_LINK,RELATED_PERATURAN_LINK,PERATURAN_jenis_peraturan,PERATURAN_pemrakarsa,PERATURAN_nomor,PERATURAN_tahun,PERATURAN_tentang,PERATURAN_tempat_penetapan,PERATURAN_status,RELATED_jenis_peraturan,RELATED_pemrakarsa,RELATED_nomor,RELATED_tahun,RELATED_tentang,RELATED_tempat_penetapan,RELATED_status
0,uu-no-13-tahun-2003,Mencabut,uu-no-25-tahun-1997,https://peraturan.go.id/id/uu-no-13-tahun-2003,https://peraturan.go.id/id/uu-no-25-tahun-1997,UNDANG-UNDANG,PEMERINTAH PUSAT,13,2003,KETENAGAKERJAAN,Jakarta,Berlaku,UNDANG-UNDANG,PEMERINTAH PUSAT,25,1997,KETENAGAKERJAAN,Jakarta,Tidak Berlaku Dicabut Oleh :Undang-Undang Nomo...


In [91]:
example_row = df_combined.head(1)
example_row["RELATED_status"]

0    Tidak Berlaku Dicabut Oleh :Undang-Undang Nomo...
Name: RELATED_status, dtype: object

In [92]:
from SPARQLWrapper import SPARQLWrapper, JSON

# Function to send SPARQL update query
def execute_sparql_update(query):
    sparql = SPARQLWrapper(UPDATE_URL) 
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setMethod("POST")

    sparql.queryAndConvert()
    print("SPARQL Update executed successfully")
    # if response.status_code != 204:
    #     print("Error:", response.status_code, response.text)
    # else:
    #     print("SPARQL Update executed successfully")

# Function to create nodes with properties
def create_node(uri, properties):
    triples = []
    considers_prop = properties['status'].replace('\n', ' ')
    triples.append(f"<{uri}> a owl:Thing ;\n")
    triples.append(f"a lexid-s:LegalDocument ;\n")
    triples.append(f"rdfs:label '{properties['peraturan']}' ;\n")
    triples.append(f"lexid-s:considers '{considers_prop}' ;\n")
    triples.append(f"lexid-s:hasRegulationNumber '{properties['nomor']}' ;\n")
    triples.append(f"lexid-s:hasRegulationYear '{properties['tahun']}' ;\n")
    triples.append(f"lexid-s:hasEnactionCity lexid:City_{properties['tempat_penetapan'].replace(' ', '_')} ;\n")
    triples.append(f"lexid-s:hasCreator lexid:Position_{properties['pemrakarsa'].replace(' ', '_')} .\n")

    query = f"INSERT DATA \n{{ {' '.join(triples)} }};\n"
    return query
    # execute_sparql_update(query)

# Function to create relationships between nodes
def create_relationship(source_uri, target_uri, relationship_type):
    query = ""
    if relationship_type == "MENCABUT":
        query += f"""
        INSERT DATA {{
            <{source_uri}> lexid-s:repeals <{target_uri}> .
        }};\n
        """
    elif relationship_type == "DICABUT":
        query += f"""
        INSERT DATA {{
            <{source_uri}> lexid-s:repealedBy <{target_uri}> .
        }};\n
        """
    elif relationship_type == "MENGUBAH":
        query += f"""
        INSERT DATA {{
            <{source_uri}> lexid-s:amends <{target_uri}> .
        }};\n
        """
    elif relationship_type == "DIUBAH":
        query += f"""
        INSERT DATA {{
            <{source_uri}> lexid-s:amendedBy <{target_uri}> .
        }};\n
        """
    return query
    # execute_sparql_update(query)

# Function to create reverse relationships
def create_reverse_relationship(source_uri, target_uri, relationship_type):
    reverse_relationship_type = ""
    if relationship_type == "MENCABUT":
        reverse_relationship_type = "DICABUT"
    elif relationship_type == "MENGUBAH":
        reverse_relationship_type = "DIUBAH"
    else:
        raise ValueError(f"Unknown relationship type: {relationship_type}")
    
    return create_relationship(target_uri, source_uri, reverse_relationship_type)

# Iterate over the DataFrame and create nodes and relationships in GraphDB
data = """
        base <http://example.com/base/>
        prefix xsd: <http://www.w3.org/2001/XMLSchema#>
        prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        prefix dbo: <http://dbpedia.org/ontology/>
        prefix dct: <http://purl.org/dc/terms/>
        prefix owl: <http://www.w3.org/2002/07/owl#>
        prefix wd: <https://www.wikidata.org/wiki/>
        prefix lexid-s: <https://w3id.org/lex-id/schema/>
        prefix lexid: <https://w3id.org/lex-id/data/>
        """
# data = ""
for index, row in df_combined.iterrows():
    # Define URIs for nodes
    peraturan_uri = f"https://w3id.org/lex-id/data/UU_{row['PERATURAN_tahun']}_{row['PERATURAN_nomor']}"
    related_peraturan_uri = f"https://w3id.org/lex-id/data/UU_{row['RELATED_tahun']}_{row['RELATED_nomor']}"

    # Define properties for the PERATURAN node
    peraturan_properties = {
        "peraturan": row['PERATURAN'],
        "id": row['PERATURAN_LINK'],
        "nomor": row['PERATURAN_nomor'],
        "jenis_peraturan": row['PERATURAN_jenis_peraturan'],
        "pemrakarsa": row['PERATURAN_pemrakarsa'],
        "tahun": row['PERATURAN_tahun'],
        "tentang": row['PERATURAN_tentang'],
        "tempat_penetapan": row['PERATURAN_tempat_penetapan'],
        "status": row['PERATURAN_status'],
    }
    data += create_node(peraturan_uri, peraturan_properties)

    # Define properties for the RELATED_PERATURAN node
    related_properties = {
        "peraturan": row['RELATED_PERATURAN'],
        "id": row['RELATED_PERATURAN_LINK'],
        "nomor": row['RELATED_nomor'],
        "jenis_peraturan": row['RELATED_jenis_peraturan'],
        "pemrakarsa": row['RELATED_pemrakarsa'],
        "tahun": row['RELATED_tahun'],
        "tentang": row['RELATED_tentang'],
        "tempat_penetapan": row['RELATED_tempat_penetapan'],
        "status": row['RELATED_status'],
    }
    data += create_node(related_peraturan_uri, related_properties)

    # Create the primary relationship and reverse relationship
    data += create_relationship(peraturan_uri, related_peraturan_uri, RELATIONSHIP_TYPE)
    data += create_reverse_relationship(peraturan_uri, related_peraturan_uri, RELATIONSHIP_TYPE)

    if index % 100 == 0:
        # print(data)
        execute_sparql_update(data)
        data = """base <http://example.com/base/>
prefix xsd: <http://www.w3.org/2001/XMLSchema#>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
prefix dbo: <http://dbpedia.org/ontology/>
prefix dct: <http://purl.org/dc/terms/>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix wd: <https://www.wikidata.org/wiki/>
prefix lexid-s: <https://w3id.org/lex-id/schema/>
prefix lexid: <https://w3id.org/lex-id/data/>
"""
        
execute_sparql_update(data)

SPARQL Update executed successfully
SPARQL Update executed successfully
SPARQL Update executed successfully
SPARQL Update executed successfully
SPARQL Update executed successfully
SPARQL Update executed successfully
SPARQL Update executed successfully


## Import Lexid-KG to Neo4J (obsolete, use GraphDB instead)

In [93]:
# !git clone https://github.com/jamessmith404/triple-uu.git

Cloning into 'triple-uu'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 11 (delta 2), reused 11 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), done.
Resolving deltas: 100% (2/2), done.


In [4]:
# import os
# import time
# from neo4j import GraphDatabase
# from rdflib import Graph as RDFGraph
# from rdflib.namespace import split_uri

# nodes = {}
# batch_size = 1000
# current_batch_count = 0

# # Define properties and relationships
# property_predicates = {
#     "rdfs:label", "dct:description", "owl:sameAs", "name", "hasRegulationNumber",
#     "hasRegulationYear", "hasCreator", "considers", "hasDictum", "hasPromulgationPlace",
#     "hasEnactionDate", "hasEnactionLocation", "hasEnactionOffice", "hasEnactionOfficial",
#     "hasPromulgationDate", "hasPromulgationLocation", "hasPromulgationOffice", "hasPromulgationOfficial"
# }

# relationship_predicates = {
#     "amendedBy", "amends", "implementedBy", "implements", "hasLegalBasis", 
#     "legalBasisOf", "repealedBy", "repeals", "hasContent", "isContentOf", 
#     "hasPart", "isPartOf"
# }

# # Initialize Neo4j driver
# driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# def commit_batch(tx):
#     """Commit the current transaction and reset the batch counter."""
#     global current_batch_count
#     tx.commit()
#     print(f"Committed {current_batch_count} operations.")
#     current_batch_count = 0

# def get_local_name(uri):
#     """Extracts the local name from a URI."""
#     try:
#         _, local_name = split_uri(uri)
#         return local_name
#     except:
#         return str(uri)

# def create_node(tx, label, node_id, uri):
#     """Create a node if it doesn't exist."""
#     if node_id not in nodes:
#         cypher_query = f"""
#         MERGE (n:{label} {{id: $node_id, uri: $uri}})
#         """
#         tx.run(cypher_query, node_id=node_id, uri=uri)
#         nodes[node_id] = True
#     return nodes[node_id]

# def set_property(tx, node_id, property_name, value):
#     """Set a property on an existing node."""
#     cypher_query = f"""
#     MATCH (n {{id: $node_id}})
#     SET n.{property_name} = $value
#     """
#     tx.run(cypher_query, node_id=node_id, value=value)

# def create_relationship(tx, source_id, target_id, relationship_type):
#     """Create a relationship between two nodes."""
#     cypher_query = f"""
#     MATCH (a {{id: $source_id}})
#     MATCH (b {{id: $target_id}})
#     MERGE (a)-[:`{relationship_type}`]->(b)
#     """
#     tx.run(cypher_query, source_id=source_id, target_id=target_id)

# def process_triple(tx, subj, pred, obj):
#     """Process a single triple and decide to create a node, set a property, or create a relationship."""
#     subj_id = get_local_name(subj)
#     pred_local = get_local_name(pred)
#     obj_id = get_local_name(obj)

#     # Check if predicate is a property
#     if pred_local in property_predicates:
#         # Create the subject node if it doesn't exist
#         create_node(tx, "Document", subj_id, str(subj))
#         # Set property on subject node
#         set_property(tx, subj_id, pred_local, str(obj))

#     # Check if predicate is a relationship
#     elif pred_local in relationship_predicates:
#         # Create both nodes if they don't exist
#         create_node(tx, "Document", subj_id, str(subj))
#         create_node(tx, "Document", obj_id, str(obj))
#         # Create relationship between the nodes
#         create_relationship(tx, subj_id, obj_id, pred_local)

# def turtle_to_cypher(tx, turtle_path):
#     global current_batch_count
#     g = RDFGraph()
#     g.parse(turtle_path, format="turtle")

#     for subj, pred, obj in g:
#         process_triple(tx, subj, pred, obj)
#         current_batch_count += 1

#         # Commit batch if limit is reached
#         if current_batch_count >= batch_size:
#             commit_batch(tx)
#             tx = driver.session().begin_transaction()

#     # Commit remaining triples
#     if current_batch_count > 0:
#         commit_batch(tx)

# # Process each Turtle file
# locations = []
# list_folder = ['Lexid/data/bn', 'Lexid/data/lain-lain', 'Lexid/data/ln', 'Lexid/data/perda']
# for folder in list_folder:
#     for (root, dirs, files) in os.walk(folder, topdown=True):
#         for file in files:
#             locations.append(os.path.join(root, file))

# start_time = time.time()
# failed_document = []
# data_to_process = len(locations)

# for file_name in locations:
#     with driver.session() as session:
#         tx = session.begin_transaction()
#         try:
#             turtle_to_cypher(tx, file_name)
#             print(f'Data from file {file_name} is successfully uploaded')
#         except Exception as e:
#             print(f'Failed to upload data from file {file_name}: {e}')
#             failed_document.append(file_name)
#             tx.rollback()
        
#         data_to_process -= 1
#         print(f"{data_to_process} files remaining")

# end_time = time.time()
# print(f"Processing Time: {end_time - start_time} seconds")

# # Report failed uploads
# print("======================FAILED DOCUMENT============================")
# print(len(failed_document))
# for i in failed_document:
#     print(i)

# with open("failed-to-upload.txt", 'w') as f:
#     for i in failed_document:
#         f.write(i)


Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
Committed 1000 operations.
C

KeyboardInterrupt: 

In [22]:
# import os
# import time
# import re
# # from py2neo import Graph, Node, Relationship
# from neo4j import GraphDatabase
# from rdflib import Graph as RDFGraph
# from rdflib.namespace import split_uri

# nodes = {}
# batch_size = 1000
# current_batch_count = 0  # Track the current number of operations in the batch

# # Initialize Neo4j driver
# # graph = Graph(NEO4J_URI, name=NEO4J_USERNAME, password=NEO4J_PASSWORD)
# driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# # Start a transaction for batch processing
# # tx = graph.begin()

# def commit_batch(tx):
#     """Commit the current transaction and reset the batch counter."""
#     global current_batch_count, graph
#     tx.commit()
#     print(f"Committed {current_batch_count} operations.")
#     # tx = graph.begin()
#     current_batch_count = 0

# # Extracts the local name from a URI, if URI malformed just return the full URI as a str
# def get_local_name(uri):
#     try:
#         prefix, local_name = split_uri(uri)
#         return local_name
#     except:
#         return str(uri)

# # Function to create nodes
# def create_node(tx, label, node_id, uri):
#     # node_id = properties['id']
#     if node_id not in nodes:
#         cypher_query = f"""
#         MERGE (n:{label} {{id: $node_id, uri: $uri}})
#         """
#         tx.run(cypher_query, node_id=node_id, uri=uri)
#         nodes[node_id] = True
#     return nodes[node_id]

# # Function to create relationships
# def create_relationship(tx, source_id, target_id, relationship_type):
#     # source_node = nodes.get(source_id)
#     # target_node = nodes.get(target_id)
#     cypher_query = f"""
#     MATCH (a {{id: $source_id}})
#     MATCH (b {{id: $target_id}})
#     MERGE (a)-[:`{relationship_type}`]->(b)
#     """
#     tx.run(cypher_query, source_id=source_id, target_id=target_id)
#     # if source_node and target_node:
#     #     relationship = Relationship(source_node, relationship_type, target_node)
#     #     graph.merge(relationship)

# # def upload_to_neo4j(cypher_query):
#     # with driver.session() as session:
#     #     session.run(cypher_query)

# def turtle_to_cypher(tx, turtle_path):
#     global current_batch_count
#     g = RDFGraph()
#     g.parse(turtle_path, format="turtle")
#     # queries = []

#     # # Start a transaction for batch processing
#     # tx = graph.begin()
    
#     # Convert RDF triples to Cypher
#     for subj, pred, obj in g:
#         subj_id = get_local_name(subj)
#         pred_local = get_local_name(pred)
#         obj_id = get_local_name(obj)

#         # Create nodes
#         create_node(tx, "Resource", subj_id, str(subj))
#         create_node(tx, "Resource", obj_id, str(obj))

#         # Create relationship
#         create_relationship(tx, subj_id, obj_id, pred_local)
#         current_batch_count += 1

#         # Commit batch if limit is reached
#         if current_batch_count >= batch_size:
#             commit_batch(tx)
#             tx = driver.session().begin_transaction()  # Start a new transaction

#         # # Add subject and object nodes if they haven't been added already
#         # if subj_id not in nodes:
#         #     subj_node = Node("Resources", id=subj_id, uri=str(subj))
#         #     nodes[subj_id] = subj_node
#         #     tx.create(subj_node)  # Add to transaction
#         #     current_batch_count += 1
            
#         # if obj_id not in nodes:
#         #     obj_node = Node("Resource", id=obj_id, uri=str(obj))
#         #     nodes[obj_id] = obj_node
#         #     tx.create(obj_node)  # Add to transaction
#         #     current_batch_count += 1

#         # Add relationship to the batch
#         # relationship = Relationship(nodes[subj_id], pred_local, nodes[obj_id])
#         # tx.create(relationship)
#         # current_batch_count += 1

#     # Commit any remaining items in the transaction
#     if current_batch_count > 0:
#         commit_batch(tx)
            
#     # Commit any remaining items in the transaction
#     # if current_batch_count > 0:

#         # # Create nodes
#         # create_node("Resource", {"id": subj_id, "uri": str(subj)})
#         # create_node("Resource", {"id": obj_id, "uri": str(obj)})
        
#         # Create relationship
#         # create_relationship(subj_id, obj_id, pred_local)

#         # # Construct Cypher query for each triple
#         # cypher_query = f"""
#         # MERGE (s:Resource {{uri: '{subj}'}})
#         # MERGE (o:Resource {{uri: '{obj}'}})
#         # MERGE (s)-[:`{pred}`]->(o)
#         # """
#         # queries.append(cypher_query)

#     # return queries
    

In [ ]:
# # Process each Turtle file
# locations = []
# list_folder = ['Lexid/data/bn', 'Lexid/data/lain-lain', 'Lexid/data/ln', 'Lexid/data/perda']
# for folder in list_folder:
#     for (root, dirs, files) in os.walk(folder, topdown=True):
#         for file in files:
#             locations.append(os.path.join(root, file))

# start_time = time.time()
# failed_document = []
# data_to_process = len(locations)

# # Upload data from each Turtle file
# for file_name in locations:
#     with driver.session() as session:
#         tx = session.begin_transaction()
#         try:
#             turtle_to_cypher(tx, file_name)
#             # cypher_queries = turtle_to_cypher(tx, file_name)
#             # for query in cypher_queries:
#             #     upload_to_neo4j(query)
#             print(f'Data from file {file_name} is successfully uploaded')
#         except Exception as e:
#             print(f'Failed to upload data from file {file_name}: {e}')
#             failed_document.append(file_name)
#             tx.rollback()
        
#         data_to_process -= 1
#         print(f"{data_to_process} files remaining")

# end_time = time.time()
# print(f"Processing Time: {end_time - start_time} seconds")

# # Report failed uploads
# print("======================FAILED DOCUMENT============================")
# print(len(failed_document))
# for i in failed_document:
#     print(i)

# with open("failed-to-upload.txt", 'w') as f:
#     for i in failed_document:
#         f.write(i)

In [ ]:
# from py2neo import Graph, Node, Relationship
# from rdflib import Graph as RDFGraph, Namespace
# from rdflib.namespace import split_uri
# from rdflib import URIRef

# # Connect to Neo4j
# graph = Graph("bolt://localhost:7687", auth=("neo4j", "your_password"))

# nodes = {}

# def get_local_name(uri):
#     try:
#         prefix, local_name = split_uri(uri)
#         return local_name
#     except:
#         return str(uri)

# # Function to create nodes
# def create_node(label, node_id, uri):
#     if node_id not in nodes:
#         node = Node(label, id=node_id, uri=str(uri))
#         graph.merge(node, label, "id")
#         nodes[node_id] = node
#     return nodes[node_id]

# # Function to create relationships
# def create_relationship(source_id, target_id, relationship_type):
#     source_node = nodes.get(source_id)
#     target_node = nodes.get(target_id)
#     if source_node and target_node:
#         relationship = Relationship(source_node, relationship_type, target_node)
#         graph.merge(relationship)

# # Parse the Turtle data
# turtle_data = """
# @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
# @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
# @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
# @prefix dbo: <http://dbpedia.org/ontology/> .
# @prefix dct: <http://purl.org/dc/terms/> .
# @prefix owl: <http://www.w3.org/2002/07/owl#> .
# @prefix wd: <https://www.wikidata.org/wiki/> .
# @prefix lexid-s: <https://w3id.org/lex-id/schema/> .
# @prefix lexid: <https://w3id.org/lex-id/data/> .
  
# lexid:Peraturan_Polri_2015_8_Section_10_1_Letter_A lexid-s:isPartOf lexid:Peraturan_Polri_2015_8_Section_10_1 .
# lexid:Peraturan_Polri_2015_8_Section_10_1_Letter_B lexid-s:isPartOf lexid:Peraturan_Polri_2015_8_Section_10_1 .
# lexid:Peraturan_Polri_2015_8_Section_10_2_Number_12_Letter_E lexid-s:isPartOf lexid:Peraturan_Polri_2015_8_Section_10_2_Number_12 .
# lexid:Peraturan_Polri_2015_8_Section_10_2_Number_3_Letter_B lexid-s:isPartOf lexid:Peraturan_Polri_2015_8_Section_10_2_Number_3 .
# """

# rdf_graph = RDFGraph()
# rdf_graph.parse(data=turtle_data, format="turtle")

# # Iterate over the triples
# for subj, pred, obj in rdf_graph:
#     subj_id = get_local_name(subj)
#     pred_local = get_local_name(pred)
#     obj_id = get_local_name(obj)
    
#     # Create nodes
#     create_node("Resource", subj_id, subj)
#     create_node("Resource", obj_id, obj)
    
#     # Create relationship
#     create_relationship(subj_id, obj_id, pred_local)
